# EHR Pipeline Walkthrough

This notebook runs the EHR path for `dre_production` from input tables through patient-level likely DRE inference, dataset profiling, baseline modeling, and reporting summaries.

`likely_dre` is a computable inference from available longitudinal patient data. It is not a formal adjudicated ILAE drug-resistant epilepsy diagnosis.

The notebook is intentionally thin. It imports package modules, calls them, and displays concise outputs. Clinical evidence construction, likely DRE rule evaluation, profiling, modeling, leakage checks, artifact registration, and summary writing live in package code.

## How to Use This Notebook

1. Update the three input paths in the input configuration section.
2. Run the pipeline execution cell to create the patient-level dataset and manifests.
3. Run the baseline modeling cell only after reviewing the leakage constraints.
4. Inspect the generated JSON summaries under `reports/ehr_walkthrough/`.

Expected input tables are generic EHR-like CSV or parquet files. They should contain a patient identifier column named `patient_id` by default. You can adjust loading requests and package configuration for local schemas.

## Environment and Path Setup

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find project root containing pyproject.toml and src/.")

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports" / "ehr_walkthrough"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "ehr_walkthrough"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Imports

In [ ]:
from datetime import UTC, datetime
import json

import pandas as pd

from core.contracts import RunContextRecord
from datasets import EHRPatientDatasetBuilder, EHRPipelineInputConfig, EHRPipelineRunner
from definitions import LikelyDREDefinition
from features import EHREvidenceBuilder
from ingestion import EHRLoader, TableLoadRequest
from insights import DatasetProfiler
from modeling import BaselineModelConfig, EHRBaselinePipeline
from reporting import ManifestWriter, RunSummaryWriter

## Run Context Setup

In [ ]:
DATASET_NAME = "ehr_patient_dataset_walkthrough"

run_context = RunContextRecord(
    run_id=f"ehr-walkthrough-{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}",
    project_name="dre_production",
    stage_name="ehr_pipeline_walkthrough",
    seed=42,
    timestamp_utc=datetime.now(UTC),
    metadata={"notebook": "ehr_pipeline_walkthrough.ipynb"},
)

pipeline_output_dir = REPORTS_DIR / "pipeline"
model_output_dir = ARTIFACTS_DIR / "baseline_model"
summary_output_dir = REPORTS_DIR / "summaries"

run_context

## Input Configuration

Replace these example paths with local extracts. The example names are placeholders and the next cell will fail clearly if they do not exist.

In [ ]:
diagnoses_path = DATA_DIR / "ehr" / "diagnoses.csv"
medications_path = DATA_DIR / "ehr" / "medications.csv"
seizure_events_path = DATA_DIR / "ehr" / "seizure_events.csv"

expected_inputs = {
    "diagnoses": diagnoses_path,
    "medications": medications_path,
    "seizure_events": seizure_events_path,
}
missing_inputs = {name: str(path) for name, path in expected_inputs.items() if not path.exists()}
if missing_inputs:
    raise FileNotFoundError(
        "Update the input paths before running the walkthrough. Missing files: "
        + json.dumps(missing_inputs, indent=2)
    )

input_config = EHRPipelineInputConfig(
    dataset_name=DATASET_NAME,
    diagnoses_request=TableLoadRequest(
        path=str(diagnoses_path),
        required_columns=["patient_id", "diagnosis_code"],
        parse_date_columns=["diagnosis_time"],
    ),
    medications_request=TableLoadRequest(
        path=str(medications_path),
        required_columns=["patient_id", "medication_name"],
        parse_date_columns=["medication_start", "medication_end"],
    ),
    seizure_events_request=TableLoadRequest(
        path=str(seizure_events_path),
        required_columns=["patient_id"],
        parse_date_columns=["event_time"],
    ),
    output_dir=str(pipeline_output_dir),
)

input_config.to_dict()

## Pipeline Execution

This cell delegates loading, evidence construction, likely DRE evaluation, dataset profiling, artifact registration, and manifest writing to package code.

In [ ]:
runner = EHRPipelineRunner()
pipeline_result = runner.run(input_config=input_config, run_context=run_context)
pipeline_result.to_dict()

## Dataset Profiling Summary

In [ ]:
patient_dataset_path = Path(pipeline_result.patient_dataset_path)
patient_df = pd.read_csv(patient_dataset_path)

dataset_profile = DatasetProfiler().profile_dataframe(
    df=patient_df,
    dataset_name=DATASET_NAME,
    patient_id_col="patient_id",
    label_col="likely_dre",
)

summary_writer = RunSummaryWriter()
dataset_profile_summary = summary_writer.build_dataset_profile_summary(
    dataset_profile,
    top_n_missing=10,
)
dataset_profile_summary.to_dict()

## Baseline Modeling Execution

The baseline model is intentionally conservative. Target-derived evidence fields are excluded to reduce leakage risk. Review the leakage exclusions before interpreting metrics.

In [ ]:
model_config = BaselineModelConfig(
    output_dir=str(model_output_dir),
    random_state=42,
)
baseline_pipeline = EHRBaselinePipeline(model_config)
training_result = baseline_pipeline.run(patient_df)
training_result.to_dict()

## Reporting Summary Generation

In [ ]:
pipeline_summary = summary_writer.build_pipeline_summary(
    run_result=pipeline_result,
    run_context=run_context,
    dataset_name=DATASET_NAME,
)
model_summary = summary_writer.build_model_summary(training_result)
reporting_bundle = summary_writer.build_reporting_bundle(
    pipeline_summary=pipeline_summary,
    model_summary=model_summary,
    dataset_profile_summary=dataset_profile_summary,
    notes=["Generated by thin EHR walkthrough notebook."],
)

pipeline_summary_path = summary_writer.write_summary_json(
    pipeline_summary,
    summary_output_dir / "pipeline_summary.json",
)
model_summary_path = summary_writer.write_summary_json(
    model_summary,
    summary_output_dir / "model_summary.json",
)
profile_summary_path = summary_writer.write_summary_json(
    dataset_profile_summary,
    summary_output_dir / "dataset_profile_summary.json",
)
bundle_path = summary_writer.write_reporting_bundle(
    reporting_bundle,
    summary_output_dir / "reporting_bundle.json",
)

{
    "pipeline_summary": str(pipeline_summary_path),
    "model_summary": str(model_summary_path),
    "dataset_profile_summary": str(profile_summary_path),
    "reporting_bundle": str(bundle_path),
}

## Results Display

In [ ]:
test_metrics = training_result.metrics.get("test", {})

results_display = {
    "patient_dataset_shape": patient_df.shape,
    "likely_dre_positive_count": pipeline_result.likely_dre_positive_count,
    "likely_dre_negative_count": pipeline_result.likely_dre_negative_count,
    "model_feature_count": len(training_result.feature_cols_used),
    "test_accuracy": test_metrics.get("accuracy"),
    "test_f1": test_metrics.get("f1"),
    "test_roc_auc": test_metrics.get("roc_auc"),
    "patient_dataset_path": pipeline_result.patient_dataset_path,
    "model_path": training_result.model_path,
    "metrics_path": training_result.metrics_path,
}
results_display

In [ ]:
artifact_paths = {
    **pipeline_summary.artifact_paths,
    "model": training_result.model_path,
    "model_metrics": training_result.metrics_path,
    "reporting_bundle": str(bundle_path),
}
{key: value for key, value in artifact_paths.items() if value is not None}

In [ ]:
{
    "top_missing_columns": dataset_profile_summary.top_missing_columns,
    "leakage_excluded_columns": model_summary.leakage_excluded_cols,
    "features_used": model_summary.feature_cols_used,
}

## Limitations and Interpretation

`likely_dre` is a computable inference from available records, not formal adjudicated ILAE DRE.

Routine EHR data may not fully establish medication tolerance, medication appropriateness, proper use, or sustained seizure freedom.

Baseline model metrics can be affected by the structure of the derived target and by leakage constraints. The modeling pipeline intentionally excludes direct evidence fields used to construct the label. This may reduce apparent performance, but it produces a more honest baseline.

This notebook covers an EHR-first path only. MRI, EEG, and multimodal production workflows are future extensions.